# Model Evaluation via Information Theory

### Objective
Measure the predictive quality of a language model using information theory. Instead of evaluating a model by looking at qualitative text outputs, we are going to implement Cross-Entropy Loss ad Perplexity (PPL) from scratch using NumPy. 
Perplexity quantifies a model's uncertaintiy, representing the geometric mean branching factor of the next-token choices 
on an unseen validation evaluation dataset.

### Core Architecture (Constraints)
- No PyTorch or Scikit-Learn Metreics: Compute all log-likelihood summaries and scaling transformations using core 
Python collections and numpy functions.

- Log-Space Safeguards: Convert your mathematical steps into natural or base-2 log calculations before averaging to prevent immediate floating-point arithmetic underflow when multiplying consecutive token probabilities.

### Engineering Requirements
1. Cross-Entropy Loss Calculator

Given an unseen test sequence of $T$ tokens and a pre-calculated probability table from an $N-gram$ model, look up the
conditional probability $P(w_t | Context)$ for each step.

- Calculate the average negative log-likelihood (Cross-Entropy Loss):
$$
Loss = -\frac{1}{T} \sum_{t=1}^{T} \log_2 P(w_t | Context)
$$

- Out-of-Vocabulary/Context Guard: If a context or target pair was completely missing form the training distribution, 
your evaluation loop must back off to the Add-Alpha background smoothing probability calculated on Day 010.

2. Perplexity Engine

Transform the cross-entropy score into the final exponential perplexity score:
$$
Perplexity = 2^Loss
$$

- Alternatively, if using natural logs (ln), compute via $exp(Loss_base-e). Perplexity represents the effective number of word your model is choosing from st any given step; a lower perplexity means the model is more confident in its predictions.

### Sample Seed Validation Data
To keep your statistical pipeline consistent, we will use the exact $N-gram$ table parameters built on Day 010 to score 
a new test string.

In [34]:
# Re-use the probability distributions computed on Day 010
# Evaluation target validation string:
test_corpus = "dense matrices represent text via structural maps"

# N-gram configuration: Trigram (N=3)
# Base vocabulary size from Day 010: |V| = 17, alpha = 0.1

### Expected Output
```
INFORMATION-THEORETIC EVALUATION (v1)

EVALUATION SYSTEM STATISTICS
Test Tokens Extracted: 7
Vocabulary Constraints Match: True

SEQUENCE COMPONENT LOSS TRACKING
Step 1: Context=('dense', 'matrices') -> Target='represent' | P=0.2340 -> Loss=2.0954
...

SUMMARY PERFORMANCE METRICS
Average Cross-Entropy Loss (Bits): ...
Final Evaluation Perplexity: ...
```

### Imports

In [35]:
from collections import defaultdict, Counter
import numpy as np

### Day 010's Probability Table Reconstruction

In [36]:
# Training Data
corpus_modeling = (
    "dense matrices capture semantic meaning via spatial proximity "
    "dense matrices capture statistical distributions via structural maps "
    "dense matrices represent text as vector coordinates"
)

tokens = corpus_modeling.split()
ngram_size = 3
alpha = 0.1

# Vocabulary
def build_vocabulary(tokens):
    "Creates vocabulary lookup tables."
    
    vocab = sorted(set(tokens))

    word_to_index = {word: i for i, word in enumerate(vocab)}
    index_to_word = {i: word for i, word in enumerate(vocab)}

    return vocab, word_to_index, index_to_word

# N-gram Context Extraction Engine
def extract_ngram_contexts(tokens, n):
    "Extract (history tuple, next word) pairs for arbitraty n-gram size."

    contexts = []
    history_size = n - 1

    for i in range(len(tokens) - history_size):
        history = tuple(tokens[i:i + history_size])
        target = tokens[i + history_size]
        contexts.append((history, target))

    return contexts

# Frequency Table Builder
def build_ngram_counts(context_pairs):
    "Builds conditional frequency tables."

    context_counts = defaultdict(Counter)

    for context, target in context_pairs:
        context_counts[context][target] += 1

    return context_counts

# Probability Distribution Engine
def compute_probability_table(context_counts, vocab, alpha = 0.1):
    "Applies Add-Alpha smoothing. P(word | context)"

    probability_table = {}
    vocab_size = len(vocab)

    for context, targets in context_counts.items():
        total = sum(targets.values())
        distribution = {}
        denominator = total + (alpha * vocab_size)

        for word in vocab:
            count = targets[word]
            probability = (count + alpha) / denominator
            distribution[word] = probability

        probability_table[context] = distribution

    return probability_table

In [37]:
# Rebuild Trained Statistical Language Model
vocab, word_to_index, index_to_word = (build_vocabulary(tokens))
context_pairs = (extract_ngram_contexts(tokens, ngram_size))
context_counts = (build_ngram_counts(context_pairs))
probability_table = (compute_probability_table(context_counts, vocab, alpha))

### Validation Dataset

In [38]:
test_corpus = ("dense matrices represent text via structural maps")
test_tokens = test_corpus.split()

### Cross Entropy Loss Engine

In [39]:
def compute_cross_entropy(tokens, prob_table, vocab, alpha = 0.1):
    "Compute cross-entropy of a sequence of tokens given a probability table."
    history_size = 2
    losses = []
    vocab_size = len(vocab)
    fallback_prob = 1.0 / vocab_size

    for i in range(len(tokens) - history_size):
        context = tuple(tokens[i:i + history_size])
        target = tokens[i + history_size]

        if context in prob_table:
            probability = prob_table[context].get(
                target, alpha / (alpha * vocab_size)
            )
        else:
            probability = fallback_prob

        loss = -np.log2(probability)
        losses.append((context, target, probability, loss))

    average_loss = (np.mean([item[3] for item in losses]))

    return average_loss, losses

### Perplexity Engine

In [40]:
def compute_perplexity(cross_entropy):
    "Compute perplexity given a cross-entropy value."
    return 2 ** cross_entropy

### Evaluation Harness

In [41]:
def evaluate_information_theory(test_tokens, prob_table, vocab):
    "Evaluate cross-entropy and perplexity for a sequence of test tokens."
    print("INFORMATION-THEORETIC EVALUATION (v1)\n")
    print("EVALUATION SYSTEM STATISTICS")
    print(f"Test Tokens Extracted: {len(test_tokens)}")
    print(f"Vocabulary Constraints Match: "
          f"{all(token in vocab for token in test_tokens)}\n"
    )

    loss, tracking = (compute_cross_entropy(test_tokens, prob_table, vocab))

    print(f"SEQUENCE COMPONENT LOSS TRACKING")

    for index, item in enumerate(tracking, start = 1):
        context, target, probability, step_loss = item

        print(
            f"Step {index}: "
            f"Context = {context} -> "
            f"Target = '{target}' | "
            f"P = {probability:.4f} -> "
            f"Loss = {step_loss:.4f}"
        )

    perplexity = (compute_perplexity(loss))

    print("\nSUMMARY PERFORMANCE METRICS")
    print(f"Average Cross-Entropy Loss (Bits): {loss:.4f}")
    print(f"Final Evaluation Perplexity (PPX): {perplexity:.4f}")

### Execute Pipeline

In [42]:
evaluate_information_theory(test_tokens, probability_table, vocab)

INFORMATION-THEORETIC EVALUATION (v1)

EVALUATION SYSTEM STATISTICS
Test Tokens Extracted: 7
Vocabulary Constraints Match: True

SEQUENCE COMPONENT LOSS TRACKING
Step 1: Context = ('dense', 'matrices') -> Target = 'represent' | P = 0.2340 -> Loss = 2.0952
Step 2: Context = ('matrices', 'represent') -> Target = 'text' | P = 0.4074 -> Loss = 1.2955
Step 3: Context = ('represent', 'text') -> Target = 'via' | P = 0.0370 -> Loss = 4.7549
Step 4: Context = ('text', 'via') -> Target = 'structural' | P = 0.0588 -> Loss = 4.0875
Step 5: Context = ('via', 'structural') -> Target = 'maps' | P = 0.4074 -> Loss = 1.2955

SUMMARY PERFORMANCE METRICS
Average Cross-Entropy Loss (Bits): 2.7057
Final Evaluation Perplexity (PPX): 6.5237
